## **read the bronze layer**

In [0]:
df_customers = spark.read.format('parquet').load('/Volumes/etl/retail/retail_files/Bronze/Customers/')

display(df_customers)

city,customer_id,email,first_name,last_name,phone,registration_date
Delhi,101,user101@example.com,Ravi,Yadav,9887654321,2023-09-14
Mumbai,102,user102@example.com,Nina,Joshi,9876543210,2024-01-21
Bangalore,103,user103@example.com,Sonal,Sharma,9865432109,2023-07-10
Hyderabad,104,user104@example.com,Karan,Patel,9854321098,2024-02-05
Chennai,105,user105@example.com,Riya,Singh,9843210987,2023-06-28
Pune,106,user106@example.com,Ajay,Mishra,9832109876,2024-03-10
Ahmedabad,107,user107@example.com,Priya,Kapoor,9821098765,2023-05-12
Kolkata,108,user108@example.com,Rahul,Verma,9810987654,2023-08-19
Delhi,109,user109@example.com,Pooja,Mehta,9809876543,2024-04-01
Mumbai,110,user110@example.com,Deepak,Nair,9798765432,2023-10-14


In [0]:
df_store=spark.read.parquet('/Volumes/etl/retail/retail_files/Bronze/stores/')
display(df_store)

store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai


In [0]:
df_trans=spark.read.parquet('/Volumes/etl/retail/retail_files/Bronze/Transactions/')
display(df_trans)

transaction_id,customer_id,product_id,store_id,quantity,transaction_date
15,121,6,5,5,2025-05-19
10,124,2,2,5,2024-08-27
2,105,3,4,5,2024-11-12
13,104,3,3,4,2025-05-04
19,116,8,4,4,2024-07-14
23,122,9,4,2,2025-04-30
5,105,5,2,1,2025-03-17
30,116,8,4,5,2025-03-16
21,105,1,3,5,2024-10-02
11,102,1,3,2,2024-08-11


In [0]:
df_products=spark.read.parquet('/Volumes/etl/retail/retail_files/Bronze/Products/')
display(df_products)

product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.00
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.00
6,Water Bottle,Fitness,299.00
7,Smartwatch,Electronics,4999.00
8,Desk Organizer,Accessories,399.00
9,Dumbbell Set,Fitness,1999.00
10,Pen Drive 32GB,Electronics,599.00


**Silver Layer**(Cleaning Data )

In [0]:
from pyspark.sql.functions import col

df_trans = df_trans.select(
    col('transaction_id').cast('int'),
    col('customer_id').cast('int'),
    col('product_id').cast('int'),   # Corrected
    col('store_id').cast('int'),
    col('quantity').cast('int'),
    col('transaction_date').cast('date')
)

In [0]:
from pyspark.sql.functions import col

df_product = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("price").cast("double"),
    col("category")
)

In [0]:
display(df_product)

product_id,product_name,price,category
1,Wireless Mouse,799.99,Electronics
2,Bluetooth Speaker,1299.49,Electronics
3,Yoga Mat,499.0,Fitness
4,Laptop Stand,999.99,Accessories
5,Notebook Set,149.0,Stationery
6,Water Bottle,299.0,Fitness
7,Smartwatch,4999.0,Electronics
8,Desk Organizer,399.0,Accessories
9,Dumbbell Set,1999.0,Fitness
10,Pen Drive 32GB,599.0,Electronics


In [0]:
df_stores=df_store.select(
    col('store_id').cast('int'),
    col('store_name'),
    col('location')
)
 

In [0]:
df_customer=df_customers.select(
    "customer_id","first_name","last_name","email","phone","city","registration_date"
)
 

In [0]:

df_silver=df_trans \
     .join(df_customer,"customer_id")\
    .join(df_products,"product_id")\
        .join(df_stores,"store_id") \
        .withColumn("total_amount",col("quantity")*col("price"))

In [0]:
display(df_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,phone,city,registration_date,product_name,category,price,store_name,location,total amount
5,6,121,15,5,2025-05-19,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,Water Bottle,Fitness,299.00,Mega Plaza,Chennai,1495.00
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Yoga Mat,Fitness,499.00,Downtown Mini Store,Pune,2495.00
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.00,Tech World Outlet,Bangalore,1996.00
5,6,121,15,5,2025-05-19,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,Water Bottle,Fitness,299.00,Mega Plaza,Chennai,1495.00
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Yoga Mat,Fitness,499.00,Downtown Mini Store,Pune,2495.00
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.00,Tech World Outlet,Bangalore,1996.00
4,8,116,19,4,2024-07-14,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Desk Organizer,Accessories,399.00,Downtown Mini Store,Pune,1596.00
4,9,122,23,2,2025-04-30,Tina,Kapoor,user122@example.com,9676543210,Pune,2023-09-20,Dumbbell Set,Fitness,1999.00,Downtown Mini Store,Pune,3998.00


In [0]:
df_silverlayer='/Volumes/etl/retail/retail_files/Silver/'
df_silver.write.mode('overwrite').format('delta').save(df_silverlayer)

In [0]:
/*Error aagya tha beacause its need a adls location */
spark.sql("""
CREATE TABLE etl.retail.silver_cleaned
USING DELTA
LOCATION '/Volumes/etl/retail/retail_files/Silver/'
""")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8829919920240305>, line 1
----> 1 spark.sql(f"""
      2 CREATE TABLE etl.retail.silver_cleaned
      3 USING DELTA
      4 LOCATION '/Volumes/etl/retail/retail_files/Silver/'
      5 """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_re

In [0]:
/* toh ham aise bhi table bana skte hai */
df_silver.write.mode('overwrite').format('delta').saveAsTable('etl.retail.silver_cleaned')

In [0]:
%sql
describe extended etl.retail.silver_cleaned

col_name,data_type,comment
store_id,int,null
product_id,int,null
customer_id,int,null
transaction_id,int,null
quantity,int,null
transaction_date,date,null
first_name,string,null
last_name,string,null
email,string,null
phone,string,null


In [0]:
%sql
select * from etl.retail.silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,phone,city,registration_date,product_name,category,price,store_name,location,total_amount
2,2,116,3,3,2025-05-01,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,3898.47
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,9709876543,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.00,Downtown Mini Store,Pune,3998.00
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.97
5,8,109,17,5,2024-07-10,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01,Desk Organizer,Accessories,399.00,Mega Plaza,Chennai,1995.00
5,6,121,15,5,2025-05-19,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,Water Bottle,Fitness,299.00,Mega Plaza,Chennai,1495.00
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Yoga Mat,Fitness,499.00,Downtown Mini Store,Pune,2495.00
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.00,Tech World Outlet,Bangalore,1996.00
4,8,116,19,4,2024-07-14,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Desk Organizer,Accessories,399.00,Downtown Mini Store,Pune,1596.00
4,9,122,23,2,2025-04-30,Tina,Kapoor,user122@example.com,9676543210,Pune,2023-09-20,Dumbbell Set,Fitness,1999.00,Downtown Mini Store,Pune,3998.00


In [0]:
#Load Cleaned transaction from silver layer
df_silver=spark.read.format('delta').load('/Volumes/etl/retail/retail_files/Silver/')

In [0]:
display(df_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,phone,city,registration_date,product_name,category,price,store_name,location,total_amount
2,2,116,3,3,2025-05-01,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,3898.47
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,9709876543,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.00,Downtown Mini Store,Pune,3998.00
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.97
5,8,109,17,5,2024-07-10,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01,Desk Organizer,Accessories,399.00,Mega Plaza,Chennai,1995.00
5,6,121,15,5,2025-05-19,Sanjay,Patel,user121@example.com,9687654321,Chennai,2024-01-10,Water Bottle,Fitness,299.00,Mega Plaza,Chennai,1495.00
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,3,105,2,5,2024-11-12,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Yoga Mat,Fitness,499.00,Downtown Mini Store,Pune,2495.00
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.00,Tech World Outlet,Bangalore,1996.00
4,8,116,19,4,2024-07-14,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Desk Organizer,Accessories,399.00,Downtown Mini Store,Pune,1596.00
4,9,122,23,2,2025-04-30,Tina,Kapoor,user122@example.com,9676543210,Pune,2023-09-20,Dumbbell Set,Fitness,1999.00,Downtown Mini Store,Pune,3998.00


In [0]:
from pyspark.sql.functions import sum,countDistinct,avg
df_gold=df_silver.groupBy("category","transaction_date","product_id","product_name","store_id","store_name","location").agg(sum("quantity").alias("total_quantity"), sum("total_amount").alias("total_amount"),countDistinct("customer_id").alias("total_customers"),)

In [0]:
display(df_gold)

category,transaction_date,product_id,product_name,store_id,store_name,location,total_quantity,total_amount,total_customers
Electronics,2024-10-08,1,Wireless Mouse,3,Tech World Outlet,Bangalore,4,3199.96,1
Accessories,2024-12-13,8,Desk Organizer,4,Downtown Mini Store,Pune,10,3990.00,1
Fitness,2025-06-03,9,Dumbbell Set,4,Downtown Mini Store,Pune,4,7996.00,1
Fitness,2024-11-12,3,Yoga Mat,4,Downtown Mini Store,Pune,10,4990.00,1
Electronics,2025-06-08,7,Smartwatch,5,Mega Plaza,Chennai,4,19996.00,1
Electronics,2025-01-04,7,Smartwatch,3,Tech World Outlet,Bangalore,10,49990.00,1
Accessories,2025-03-16,8,Desk Organizer,4,Downtown Mini Store,Pune,10,3990.00,1
Stationery,2025-03-17,5,Notebook Set,2,High Street Store,Delhi,2,298.00,1
Fitness,2025-05-04,3,Yoga Mat,3,Tech World Outlet,Bangalore,8,3992.00,1
Fitness,2024-11-29,6,Water Bottle,2,High Street Store,Delhi,8,2392.00,1


In [0]:
gold_path='/Volumes/etl/retail/retail_files/Gold/'
df_gold.write.mode('overwrite').format('delta').save(gold_path)

In [0]:

df_gold.write.mode('overwrite').format('delta').saveAsTable('etl.retail.gold_cleaned')